In [11]:

#Imports
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.data import load_matches, load_elo_baseline
from src.elo import prepare_matches, run_elo_updates, update_elo, tournament_weight
from src.features import get_stats, update_team_history, get_h2h_stat, update_h2h
from collections import deque,defaultdict
import joblib

import pandas as pd

sys.path.append(os.path.abspath(".."))



In [12]:
# Load data
df = load_matches() #Load dataframe containing all historical matches
country_elo = load_elo_baseline() #Generate a dictionary with every country having their elo from 12/13/2025

# Prepare and update
recent_matches = prepare_matches(df, start_date="2025-12-13", end_date="2026-06-11") #We need these games since the current elo is only accurate up til 12/13/2025
country_elo = run_elo_updates(recent_matches, country_elo)

#Top 20
top20 = sorted(country_elo.items(), key=lambda x: x[1], reverse=True)[:20]
print("Top 20 teams going into the WC based off ELO")
for rank, (team, rating) in enumerate(top20, 1):
    print(f"{rank}. {team}: {rating:.0f}")

Top 20 teams going into the WC based off ELO
1. Spain: 2154
2. Argentina: 2115
3. France: 2062
4. England: 2021
5. Brazil: 1993
6. Portugal: 1987
7. Colombia: 1981
8. Netherlands: 1944
9. Ecuador: 1939
10. Germany: 1933
11. Norway: 1916
12. Croatia: 1912
13. Turkey: 1911
14. Japan: 1906
15. Belgium: 1892
16. Uruguay: 1892
17. Switzerland: 1890
18. Morocco: 1878
19. Mexico: 1876
20. Italy: 1868


ELO caculations turn out essentially the same to the elo rating found on https://www.eloratings.net/2026_World_Cup, 

In [13]:
current_elos = defaultdict(lambda: 1500)
team_history = defaultdict(lambda: deque(maxlen=10))
h2h_history = defaultdict(lambda: deque(maxlen=5))
training_rows = []

df_sorted = prepare_matches(df, df['date'].min(), pd.Timestamp("2026-06-11")).sort_values('date')
for row in df_sorted.itertuples(index=False):
    if row.tournament == 'Friendly': #Don't add friendlies
        continue
    home_elo = current_elos[row.home_team]
    away_elo = current_elos[row.away_team]
    
    h_form, h_gd = get_stats(row.home_team, team_history)
    a_form, a_gd = get_stats(row.away_team, team_history)
    h2h_val = get_h2h_stat(row.home_team, row.away_team, h2h_history)
        
    training_rows.append({
        'date': row.date,
        'home_team': row.home_team,
        'away_team': row.away_team,
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo-away_elo,
        "home_score": row.home_score,
        "away_score": row.away_score,
        'home_form': h_form,
        'away_form': a_form,
        'h2h': h2h_val,
        'home_gd': h_gd,
        'away_gd': a_gd,
        'neutral': 1 if row.neutral else 0,
        'K_factor': row.K_factor,
        'target': row.result,
        'tournament_weight': tournament_weight(row.tournament)
    })
    
    new_away, new_home = update_elo(
        row.result, 
        row.neutral, 
        row.K_factor, 
        row.home_score, 
        row.away_score, 
        current_elos[row.away_team], 
        current_elos[row.home_team]
    )
    
    current_elos[row.home_team] = new_home
    current_elos[row.away_team] = new_away
    
    update_team_history(row, row.home_score, row.away_score, team_history)
    update_h2h(row, h2h_history)
    
# 3. Finalize
train_df = pd.DataFrame(training_rows)
train_df=train_df.round(2)
train_df.to_csv("/Users/armand_k/Projects/WC 2026/data/world_cup_training_data.csv", index=False)

/var/folders/f0/0r4c9w2d0vx1d7zk05qxcr4h0000gn/T/ipykernel_2079/3767549619.py:55: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  train_df=train_df.round(2)


In [14]:
#WC 2026 data
country_elo

wc_groups = {
    "A": ["Mexico", "South Korea", "South Africa", "Czech Republic"],
    "B": ["Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

roster_rows = []

for group_name, teams in wc_groups.items():
    for team in teams:
        elo = country_elo[team]
        form, gd = get_stats(team, team_history)
        
        roster_rows.append({
            "Team": team,
            "Group": group_name.split()[-1],
            "elo": round(elo,2),
            "Avg ppg (L10)": form,
            "Avg gd (L10)": gd,}
        )
roster_df = pd.DataFrame(roster_rows)
roster_df.to_csv("/Users/armand_k/Projects/WC 2026/data/wc_2026_teams.csv", index=False)

In [16]:
joblib.dump(dict(country_elo), "final_elo.joblib")

['final_elo.joblib']